# Evaluación de embeddings para recuperación semántica

Este notebook compara dos modelos multilingües de embeddings utilizados durante el desarrollo original del sistema RAG:

- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- `intfloat/multilingual-e5-small`

La evaluación se concentra en la etapa de **retrieval**. El objetivo es medir si, ante consultas reformuladas, el sistema recupera la pregunta esperada de la base de conocimiento dentro de los primeros resultados Top-K.

> Esta evaluación no mide la calidad de generación de FLAN-T5. Separa deliberadamente retrieval y generación para poder analizar el componente de embeddings de forma independiente.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import EvaluationCase, compare_embedding_models

## Casos de evaluación

Se utilizan consultas reformuladas respecto de las preguntas almacenadas. Para cada consulta se define cuál es la pregunta de la base de conocimiento que debería ser recuperada.

El conjunto es pequeño y está pensado como una evaluación reproducible del prototipo, no como un benchmark general de modelos de embeddings.

In [ ]:
test_cases = [
    EvaluationCase(
        query="Necesito cambiar la fecha de mi cita, ¿qué hago?",
        expected_question="¿Puedo reprogramar mi turno?",
    ),
    EvaluationCase(
        query="¿Qué información tengo que mandar para reservar una consulta?",
        expected_question="¿Qué datos necesito para pedir un turno?",
    ),
    EvaluationCase(
        query="Tengo una emergencia, ¿me pueden atender ahora?",
        expected_question="¿Atienden urgencias?",
    ),
    EvaluationCase(
        query="¿Aceptan pacientes sin cobertura médica?",
        expected_question="¿Puedo atenderme de forma particular?",
    ),
    EvaluationCase(
        query="¿Puedo mandar mis análisis por WhatsApp para que los revisen?",
        expected_question="¿Puedo consultar resultados de estudios por WhatsApp?",
    ),
    EvaluationCase(
        query="¿Con qué medios puedo abonar la consulta?",
        expected_question="¿Cuáles son las formas de pago?",
    ),
    EvaluationCase(
        query="¿Qué sucede si aparezco después de la hora acordada?",
        expected_question="¿Qué pasa si llego tarde al turno?",
    ),
    EvaluationCase(
        query="¿Conviene llevar análisis e informes que ya tengo?",
        expected_question="¿Debo llevar estudios anteriores?",
    ),
    EvaluationCase(
        query="¿Cómo aviso que finalmente voy a asistir a la cita?",
        expected_question="¿Cómo confirmo mi turno?",
    ),
    EvaluationCase(
        query="¿Me pueden emitir un comprobante por la atención?",
        expected_question="¿Entregan factura?",
    ),
]

len(test_cases)

## Comparación MiniLM vs E5

Se evalúa la tasa de aciertos para `Top-1`, `Top-3` y `Top-5`. En los modelos E5, el módulo de evaluación aplica los prefijos `query:` y `passage:` de la misma forma que el chatbot principal.

In [ ]:
embedding_models = [
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-small",
]

summary, details = compare_embedding_models(
    knowledge_base_path=ROOT / "data" / "knowledge_base.csv",
    embedding_models=embedding_models,
    cases=test_cases,
    top_k_values=(1, 3, 5),
)

summary

## Análisis de errores

La tabla de detalle permite identificar consultas en las que la pregunta esperada no aparece dentro del Top-K. También muestra el primer resultado recuperado y su distancia coseno en Chroma.

In [ ]:
errors = details.loc[
    ~details["hit"],
    [
        "model",
        "query",
        "expected_question",
        "top_k",
        "expected_rank",
        "top_result",
        "top_distance",
    ],
]

errors

## Criterio de interpretación

Para este proyecto se prioriza:

1. Mayor `hit_rate` con valores bajos de Top-K.
2. Menor `mean_expected_rank`, de modo que el contexto relevante aparezca lo antes posible.
3. Revisión cualitativa de los errores de recuperación.

La selección de un modelo no debería basarse únicamente en una diferencia pequeña dentro de este conjunto de diez consultas. La calidad y cobertura de la base de conocimiento también condicionan directamente el desempeño del sistema RAG.